# Kiểm thử API NLIProvider (BamiBERT-ViLegalNLI)

- **Thư mục root**: `ML_final`
- **Dữ liệu đánh giá**: `src/db/eval/vlsp_nli.parquet`

Notebook này kiểm thử trực tiếp các endpoints của **FastAPI NLIProvider** bằng `TestClient` (không cần chạy uvicorn server riêng).

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

# Đảm bảo ML_final nằm trong sys.path và là working directory
CURRENT_DIR = Path.cwd()
for p in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if p.name == "ML_final" or (p / "src" / "be").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        os.chdir(str(p))
        break

print(f"📁 Thư mục làm việc hiện tại: {Path.cwd()}")

from fastapi.testclient import TestClient
from src.be.NLIProvider.api import app

client = TestClient(app)
print("✅ Đã khởi tạo TestClient thành công!")

## 1. Kiểm tra trạng thái hệ thống (`GET /health`)

In [ ]:
response = client.get("/health")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

## 2. Dự đoán NLI đơn lẻ (`POST /predict`)

Dự đoán quan hệ suy luận cho 1 cặp câu hỏi / nhận định và văn bản pháp luật kèm metadata lưu chat history.

In [ ]:
single_payload = {
    "specific_question": "Người sử dụng đất có quyền chuyển nhượng quyền sử dụng đất không?",
    "legal_document": "Căn cứ Điều 167 Luật Đất đai 2024, người sử dụng đất được thực hiện các quyền chuyển đổi, chuyển nhượng, cho thuê, cho thuê lại, thừa kế, tặng cho quyền sử dụng đất theo quy định của pháp luật.",
    "id": "item_001",
    "session_id": "session_chat_demo_123",
    "metadata": {
        "user_id": "vinh_01",
        "source": "chatbot_ui"
    }
}

res_single = client.post("/predict", json=single_payload)
print(f"Status Code: {res_single.status_code}")
result = res_single.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

## 3. Dự đoán NLI theo lô (`POST /predict_batch`)

Xử lý đồng thời danh sách nhiều cặp câu hỏi và trích đoạn luật.

In [ ]:
batch_payload = {
    "items": [
        {
            "id": "sample_1",
            "specific_question": "Người Việt Nam định cư ở nước ngoài được sở hữu nhà ở tại Việt Nam.",
            "legal_document": "Luật Đất đai 2024 quy định người gốc Việt Nam định cư ở nước ngoài được phép sở hữu nhà ở gắn liền với quyền sử dụng đất tại Việt Nam."
        },
        {
            "id": "sample_2",
            "specific_question": "Hạn mức giao đất nông nghiệp không được vượt quá 1000 ha cho cá nhân.",
            "legal_document": "Hạn mức giao đất trồng cây hàng năm, đất nuôi trồng thủy sản cho cá nhân không quá 03 héc ta cho mỗi loại đất."
        }
    ],
    "session_id": "batch_test_001"
}

res_batch = client.post("/predict_batch", json=batch_payload)
print(f"Status Code: {res_batch.status_code}")
print(json.dumps(res_batch.json(), indent=2, ensure_ascii=False))

## 4. Đánh giá tập dữ liệu chuẩn từ file Parquet (`POST /evaluate`)

Đọc trực tiếp 10 mẫu đầu từ file `src/db/eval/vlsp_nli.parquet` và gửi vào endpoint `/evaluate` để kiểm tra độ chính xác và tính metrics.

In [ ]:
eval_data_path = Path("src/db/eval/vlsp_nli.parquet")
df_eval = pd.read_parquet(eval_data_path)

# Lấy 10 mẫu thử nghiệm từ dataset
eval_samples = df_eval[["specific_question", "legal_document", "answer"]].head(10).to_dict(orient="records")

res_eval = client.post("/evaluate", json=eval_samples)
print(f"Status Code: {res_eval.status_code}")
print(json.dumps(res_eval.json(), indent=2, ensure_ascii=False))